# MVP — Pontos de Cultura nos municípios do RJ

**Execute as células em ordem no Databricks Free Edition.** Este arquivo contém código e instruções; os resultados e as capturas de tela só existirão depois da execução na sua conta.

**Pergunta:** como os Pontos de Cultura presentes no CSV da Rede Cultura Viva se distribuem entre os municípios do RJ?

**Fonte:** Ministério da Cultura, `pontosdecultura-redeculturaviva.csv` ([página da base](https://dados.cultura.gov.br/dataset/pontos-de-cultura)); licença indicada no portal: Creative Commons Atribuição. Referência municipal: [API de Localidades do IBGE](https://servicodados.ibge.gov.br/api/v1/localidades/estados/33/municipios), consulta de 22/09/2026.

**Privacidade:** o CSV original contém nome, CPF e contato de pessoas. Envie ao Databricks somente `Pontos_Cultura_campos_territoriais.csv`, uma cópia com quatro campos, sem dados pessoais diretos. Não publique o original no GitHub.

## 1. Coleta e carga — informe onde o arquivo foi enviado

Faça upload da cópia reduzida `Pontos_Cultura_campos_territoriais.csv` a um **Volume** no Databricks e copie o caminho completo que começa com `/Volumes/`. Preencha o catálogo e o esquema existentes na sua conta; o notebook não cria nem altera esses espaços. A tabela `bronze` abaixo guarda as colunas necessárias sem modificar seus valores, enquanto o CSV original permanece fora da plataforma.

In [0]:
CAMINHO_CSV = "/Volumes/workspace/default/mvp_dados-pontos_de_cultura/Pontos_Cultura_campos_territoriais.csv"
CATALOGO = "workspace"
ESQUEMA = "default"

assert "SEU_" not in CAMINHO_CSV + CATALOGO + ESQUEMA, "Preencha os três campos acima antes de executar."
from pathlib import Path
assert Path(CAMINHO_CSV).is_file(), "Arquivo não encontrado; confira o caminho copiado do Volume."
PREFIXO = f"{CATALOGO}.{ESQUEMA}"
print("Arquivo encontrado. Destino das tabelas:", PREFIXO)

Arquivo encontrado. Destino das tabelas: workspace.default


## 2. Bronze — preservação dos valores recebidos

A leitura considera separador `;` e codificação `utf-8-sig`, identificados na inspeção do arquivo. Selecionamos apenas os campos necessários. **Id não é CPF**: é o identificador do registro. Não imprimimos linhas individuais neste notebook.

In [0]:
import csv
from pyspark.sql import functions as F, types as T

CAMPOS = {"Id": "id_original", "Estado": "estado_original", "Município": "municipio_original", "Tipo de agente da Rede Cultura Viva": "tipo_agente_original"}
registros = []
with open(CAMINHO_CSV, "r", encoding="utf-8-sig", newline="") as arquivo:
    leitor = csv.DictReader(arquivo, delimiter=";")
    assert leitor.fieldnames is not None and len(leitor.fieldnames) == 4, "Estrutura do CSV mudou; confira antes de continuar."
    assert set(CAMPOS).issubset(leitor.fieldnames), "Uma coluna esperada está ausente."
    for linha in leitor:
        assert None not in linha and None not in linha.values(), "Linha CSV incompleta ou com colunas extras."
        registros.append(tuple(linha[c] for c in CAMPOS))

schema_bronze = T.StructType([T.StructField(v, T.StringType(), True) for v in CAMPOS.values()])
bronze = spark.createDataFrame(registros, schema_bronze)
assert bronze.count() == len(registros)
bronze.write.format("delta").mode("overwrite").saveAsTable(f"{PREFIXO}.bronze_pontos_cultura")
print("Registros originais:", len(registros), "| Registros com Estado RJ:", bronze.filter(F.upper(F.trim("estado_original")) == "RJ").count())

Registros originais: 7257 | Registros com Estado RJ: 789


## 3. Silver — verificar e padronizar localidades

Usamos a lista oficial do IBGE incorporada neste notebook como referência de nomes e códigos. A coluna `regra_tratamento` permite explicar cada alteração. `Guarulhos (SP)` com UF `RJ` permanece marcado como conflito, sem atribuição automática a município fluminense.

In [0]:
MUNICIPIOS_IBGE_RJ = {'Angra dos Reis': 3300100, 'Aperibé': 3300159, 'Araruama': 3300209, 'Areal': 3300225, 'Armação dos Búzios': 3300233, 'Arraial do Cabo': 3300258, 'Barra do Piraí': 3300308, 'Barra Mansa': 3300407, 'Belford Roxo': 3300456, 'Bom Jardim': 3300506, 'Bom Jesus do Itabapoana': 3300605, 'Cabo Frio': 3300704, 'Cachoeiras de Macacu': 3300803, 'Cambuci': 3300902, 'Carapebus': 3300936, 'Comendador Levy Gasparian': 3300951, 'Campos dos Goytacazes': 3301009, 'Cantagalo': 3301108, 'Cardoso Moreira': 3301157, 'Carmo': 3301207, 'Casimiro de Abreu': 3301306, 'Conceição de Macabu': 3301405, 'Cordeiro': 3301504, 'Duas Barras': 3301603, 'Duque de Caxias': 3301702, 'Engenheiro Paulo de Frontin': 3301801, 'Guapimirim': 3301850, 'Iguaba Grande': 3301876, 'Itaboraí': 3301900, 'Itaguaí': 3302007, 'Italva': 3302056, 'Itaocara': 3302106, 'Itaperuna': 3302205, 'Itatiaia': 3302254, 'Japeri': 3302270, 'Laje do Muriaé': 3302304, 'Macaé': 3302403, 'Macuco': 3302452, 'Magé': 3302502, 'Mangaratiba': 3302601, 'Maricá': 3302700, 'Mendes': 3302809, 'Mesquita': 3302858, 'Miguel Pereira': 3302908, 'Miracema': 3303005, 'Natividade': 3303104, 'Nilópolis': 3303203, 'Niterói': 3303302, 'Nova Friburgo': 3303401, 'Nova Iguaçu': 3303500, 'Paracambi': 3303609, 'Paraíba do Sul': 3303708, 'Paraty': 3303807, 'Paty do Alferes': 3303856, 'Petrópolis': 3303906, 'Pinheiral': 3303955, 'Piraí': 3304003, 'Porciúncula': 3304102, 'Porto Real': 3304110, 'Quatis': 3304128, 'Queimados': 3304144, 'Quissamã': 3304151, 'Resende': 3304201, 'Rio Bonito': 3304300, 'Rio Claro': 3304409, 'Rio das Flores': 3304508, 'Rio das Ostras': 3304524, 'Rio de Janeiro': 3304557, 'Santa Maria Madalena': 3304607, 'Santo Antônio de Pádua': 3304706, 'São Francisco de Itabapoana': 3304755, 'São Fidélis': 3304805, 'São Gonçalo': 3304904, 'São João da Barra': 3305000, 'São João de Meriti': 3305109, 'São José de Ubá': 3305133, 'São José do Vale do Rio Preto': 3305158, 'São Pedro da Aldeia': 3305208, 'São Sebastião do Alto': 3305307, 'Sapucaia': 3305406, 'Saquarema': 3305505, 'Seropédica': 3305554, 'Silva Jardim': 3305604, 'Sumidouro': 3305703, 'Tanguá': 3305752, 'Teresópolis': 3305802, 'Trajano de Moraes': 3305901, 'Três Rios': 3306008, 'Valença': 3306107, 'Varre-Sai': 3306156, 'Vassouras': 3306206, 'Volta Redonda': 3306305}


In [0]:
import re, unicodedata
from collections import Counter

def normalizar_nome(valor):
    valor = re.sub(r"\s*\(RJ\)\s*", "", valor or "", flags=re.I)
    return " ".join("".join(c for c in unicodedata.normalize("NFKD", valor) if not unicodedata.combining(c)).casefold().split())

referencia = {normalizar_nome(nome): (nome, codigo) for nome, codigo in MUNICIPIOS_IBGE_RJ.items()}
# Correspondências conferidas na lista de distritos do IBGE; nomes entre parênteses trazem o próprio município.
DISTRITOS = {"rosal":"Bom Jesus do Itabapoana", "barra de sao joao":"Casimiro de Abreu", "sana":"Macaé", "conservatoria":"Valença", "purilandia (porciuncula)":"Porciúncula", "bacaxa (saquarema)":"Saquarema", "arrozal (pirai)":"Piraí", "corrego do ouro (macae)":"Macaé", "barao de juparana (valenca)":"Valença"}
GRAFIAS = {"nova iguacun":"Nova Iguaçu", "angra do reis":"Angra dos Reis", "parati":"Paraty"}

saida = []
for linha in bronze.filter(F.upper(F.trim("estado_original")) == "RJ").collect():
    original = linha["municipio_original"] or ""
    chave = normalizar_nome(original)
    if chave in referencia:
        nome, codigo = referencia[chave]
        regra = "nome_oficial" if original.strip() == nome else "padronizacao_grafia"
    elif chave in DISTRITOS:
        nome = DISTRITOS[chave]; codigo = MUNICIPIOS_IBGE_RJ[nome]; regra = "distrito_para_municipio"
    elif chave in GRAFIAS:
        nome = GRAFIAS[chave]; codigo = MUNICIPIOS_IBGE_RJ[nome]; regra = "correcao_grafia"
    else:
        nome = None; codigo = None; regra = "localizacao_conflitante" if "(SP)" in original.upper() else "revisao_manual"
    saida.append((int(linha["id_original"]), linha["estado_original"], original, linha["tipo_agente_original"], nome, codigo, regra))

schema_silver = "id_registro long, estado_original string, municipio_original string, tipo_agente string, municipio_ibge string, codigo_ibge long, regra_tratamento string"
silver = spark.createDataFrame(saida, schema_silver)
assert silver.count() == 789, "A quantidade RJ mudou; inspecione a fonte e as regras."
assert silver.select("id_registro").distinct().count() == silver.count(), "Há IDs repetidos no RJ."
silver.write.format("delta").mode("overwrite").saveAsTable(f"{PREFIXO}.silver_pontos_rj")
print("Distribuição das regras:")
silver.groupBy("regra_tratamento").count().orderBy("regra_tratamento").show(truncate=False)
print("Registros sem município oficial:", silver.filter(F.col("codigo_ibge").isNull()).count())

Distribuição das regras:
+-----------------------+-----+
|regra_tratamento       |count|
+-----------------------+-----+
|correcao_grafia        |7    |
|distrito_para_municipio|9    |
|localizacao_conflitante|2    |
|nome_oficial           |622  |
|padronizacao_grafia    |149  |
+-----------------------+-----+

Registros sem município oficial: 2


## 4. Gold — resposta pronta para consulta

Uma linha por município validado. Não crie linhas com zero para municípios ausentes: ausência no arquivo não demonstra ausência de Pontos de Cultura.

In [0]:
gold = (silver.filter(F.col("codigo_ibge").isNotNull())
        .groupBy("codigo_ibge", "municipio_ibge")
        .agg(F.countDistinct("id_registro").alias("quantidade_pontos")))
assert gold.agg(F.sum("quantidade_pontos")).first()[0] == silver.filter(F.col("codigo_ibge").isNotNull()).count()
gold.write.format("delta").mode("overwrite").saveAsTable(f"{PREFIXO}.gold_pontos_por_municipio")
print("Registros territoriais validados:", gold.agg(F.sum("quantidade_pontos")).first()[0])
print("Municípios presentes neste arquivo:", gold.count())
print("Maiores contagens neste arquivo:")
gold.orderBy(F.desc("quantidade_pontos"), "municipio_ibge").show(10, truncate=False)

Registros territoriais validados: 787
Municípios presentes neste arquivo: 76
Maiores contagens neste arquivo:
+-----------+------------------+-----------------+
|codigo_ibge|municipio_ibge    |quantidade_pontos|
+-----------+------------------+-----------------+
|3304557    |Rio de Janeiro    |358              |
|3303302    |Niterói           |37               |
|3301702    |Duque de Caxias   |34               |
|3303500    |Nova Iguaçu       |33               |
|3304904    |São Gonçalo       |23               |
|3303906    |Petrópolis        |20               |
|3300456    |Belford Roxo      |16               |
|3305109    |São João de Meriti|16               |
|3303401    |Nova Friburgo     |13               |
|3306305    |Volta Redonda     |13               |
+-----------+------------------+-----------------+
only showing top 10 rows


## 5. Evidências e limites — preencher depois da execução

Tire capturas: (1) cópia reduzida no Volume; (2) as três tabelas persistidas no catálogo; (3) contagens de Bronze/Silver, regras de Silver e resultado Gold; (4) consulta da tabela Gold. Acrescente as capturas ao README/PDF do repositório. Não publique o CSV, imagens de dados pessoais nem tokens. A data de criação do cadastro não representa o início das atividades culturais; contagens deste arquivo não descrevem necessariamente a rede atual.